### Multi vector retriever for RAG on tables and texts using LangChain.

### LangChain Semi-Structured RAG

## INGESTION

1. Load Documents
2. Split into Chunks
3. Create a Vector Store

In [36]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

### Choose LLM you want to work with

In [37]:
# Initialize the local Ollama model
model = ChatOllama(model="llama3", temperature=0)

In [38]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers in exactly one sentence."),
    ("human", "{question}")
])

# Chain them together and invoke
chain = prompt | model
response = chain.invoke({"question": "What are the leading vector databases in the market?"})

print(response.content)

The leading vector databases in the market include Amazon SageMaker Vector Database, Databricks MLflow, Hugging Face Transformers, and Turi Create, which offer scalable and efficient storage and processing of large-scale vector data for various applications.


In [39]:
import pymupdf4llm
import pymupdf

In [40]:
import pandas as pd

In [41]:
pdf_path = "/Users/jyotisingh/Downloads/financial_annual_report.pdf"

### Extract PDF to Markdown

Raw PDF text extraction is noisy and loses layout info. Converting to Markdown preserves structural elements like headers (#, ##), lists, and tables.

In [42]:
# To retrieve your document content in Markdown, use the to_markdown() method - 
# Extract markdown text from all pages

md = pymupdf4llm.to_markdown(pdf_path)

In [43]:
print(md[:-300])

# **FY2025 ANNUAL FINANCIAL REPORT** 

Performance & Financial Statements For Three Growth-Stage Ventures 

**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data 

Fictitious Startups — Combined Annual Report FY2025 

## **Executive Portfolio Overview** 

This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-line growth, margin expansion, and a systematic path toward capital efficiency. 

## **1. NovaQuant AI** 

**Sector:** B2B Enterprise AI & Machine Learning Infrastructure 

NovaQuant AI special

In [44]:
len(md)

4913

## Load or Process via LangChain

You can directly pass the extracted Markdown text into a LangChain markdown splitter.

In [45]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [46]:
headers_to_split_on = [ ('#', 'Header 1'),
                        ('##', 'Header 2'),
                        ('###', 'Header 3'),
]

In [47]:
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

In [48]:
chunks = markdown_splitter.split_text(md)

In [49]:
print(chunks)

[Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**'}, page_content='Performance & Financial Statements For Three Growth-Stage Ventures  \n**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  \nFictitious Startups — Combined Annual Report FY2025'), Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**Executive Portfolio Overview**'}, page_content='This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-line growth, margin expansion, and a systematic pa

In [50]:
print(len(chunks))

9


### Inspecting Chunks

In [ ]:
# Inspecting the chunks

for idx, chunk in enumerate(chunks):
    print(f"--- CHUNK {idx} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Metadata_keys: {len(chunk.metadata.keys())}")
    print(f"Content:\n{chunk.page_content}\n")

--- CHUNK 0 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**'}
Metadata_keys: 1
Content:
Performance & Financial Statements For Three Growth-Stage Ventures  
**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  
Fictitious Startups — Combined Annual Report FY2025

--- CHUNK 1 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**Executive Portfolio Overview**'}
Metadata_keys: 2
Content:
This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-line growth, marg

PDF -> Markdown text -> split at headers -> Update metadata: add content_type as 'text' and 'table'

In [30]:
print(chunks[3].metadata)

{'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**1. NovaQuant AI**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**'}


In [60]:
# Updating the chunks metadata: text and tables

for idx, chunk in enumerate(chunks):
    print(f"--- CHUNK {idx} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content:\n{chunk.page_content}\n")

    # Updating chunk's metadata - content_type - text, table

    if len(chunk.metadata.keys()) == 3:
        chunk.metadata['content_type'] = 'table'
    else:
        chunk.metadata['content_type'] = 'text'

--- CHUNK 0 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**'}
Content:
Performance & Financial Statements For Three Growth-Stage Ventures  
**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  
Fictitious Startups — Combined Annual Report FY2025

--- CHUNK 1 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**Executive Portfolio Overview**'}
Content:
This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-line growth, margin expansion, and a systematic pat

In [61]:
# Inspecting chunks after updating metadata

for idx, chunk in enumerate(chunks):
    print(f"--- CHUNK {idx} ---")
    print(f"Metadata: {chunk.metadata}")
    #print(f"Metadata_keys: {len(chunk.metadata.keys())}")
    print(f"Content:\n{chunk.page_content}\n")


--- CHUNK 0 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'content_type': 'text'}
Content:
Performance & Financial Statements For Three Growth-Stage Ventures  
**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  
Fictitious Startups — Combined Annual Report FY2025

--- CHUNK 1 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**Executive Portfolio Overview**', 'content_type': 'text'}
Content:
This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-lin

### Table chunks

In [ ]:
# Inspecting table content chunks

for idx, chunk in enumerate(chunks):
    
    if chunk.metadata['content_type'] == 'table':
        print(f"--- CHUNK {idx} ---")
        print(f"Metadata: {chunk.metadata}")
        print(f"Content:\n{chunk.page_content}\n")

--- CHUNK 3 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**1. NovaQuant AI**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**', 'content_type': 'table'}
Content:
|**Metric (All figures in USD)**|**FY2024**|**FY2025**|**YoY Change (%)**|
|---|---|---|---|
|Annual Recurring Revenue (ARR)|$4,200,000|$8,450,000|+101.2%|
|Total Revenue|$5,100,000|$9,200,000|+80.4%|
|Cost of Goods Sold (COGS)|$1,122,000|$1,656,000|+47.6%|
|**Gross Profit**|**$3,978,000**|**$7,544,000**|**+89.6%**|
|Gross Margin (%)|78.0%|82.0%|+4.0%|
|Research & Development|$2,500,000|$3,800,000|+52.0%|
|Sales & Marketing|$1,800,000|$2,400,000|+33.3%|
|General & Administrative|$850,000|$1,100,000|+29.4%|
|**Net Income / (Loss)**|**($1,172,000)**|**$244,000**|**Turnaround**|

--- CHUNK 5 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**2. VerdantGro Systems**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**', 'content_type'

### Text chunks

In [63]:
# Inspecting text content chunks

for idx, chunk in enumerate(chunks):
    
    if chunk.metadata['content_type'] == 'text':
        print(f"--- CHUNK {idx} ---")
        print(f"Metadata: {chunk.metadata}")
        print(f"Content:\n{chunk.page_content}\n")

--- CHUNK 0 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'content_type': 'text'}
Content:
Performance & Financial Statements For Three Growth-Stage Ventures  
**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  
Fictitious Startups — Combined Annual Report FY2025

--- CHUNK 1 ---
Metadata: {'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**Executive Portfolio Overview**', 'content_type': 'text'}
Content:
This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-lin

**Multivector retriever**

Tables become Paragraph/structured summary.


In [64]:
# Create a dictionary to store counts of each type
category_counts = {}

for element in chunks:
    category = element.metadata['content_type']

    if category in category_counts:
         category_counts[category] += 1
    else:
        category_counts[category] = 1

unique_categories = set(category_counts.keys())

category_counts


{'text': 6, 'table': 3}

In [65]:
# Tables

table_elements = [chunk for chunk in chunks if chunk.metadata['content_type'] == 'table']
print(len(table_elements))

3


In [66]:
table_elements

[Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**1. NovaQuant AI**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**', 'content_type': 'table'}, page_content='|**Metric (All figures in USD)**|**FY2024**|**FY2025**|**YoY Change (%)**|\n|---|---|---|---|\n|Annual Recurring Revenue (ARR)|$4,200,000|$8,450,000|+101.2%|\n|Total Revenue|$5,100,000|$9,200,000|+80.4%|\n|Cost of Goods Sold (COGS)|$1,122,000|$1,656,000|+47.6%|\n|**Gross Profit**|**$3,978,000**|**$7,544,000**|**+89.6%**|\n|Gross Margin (%)|78.0%|82.0%|+4.0%|\n|Research & Development|$2,500,000|$3,800,000|+52.0%|\n|Sales & Marketing|$1,800,000|$2,400,000|+33.3%|\n|General & Administrative|$850,000|$1,100,000|+29.4%|\n|**Net Income / (Loss)**|**($1,172,000)**|**$244,000**|**Turnaround**|'),
 Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**2. VerdantGro Systems**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**', 'content_t

In [67]:
# Text

text_elements = [chunk for chunk in chunks if chunk.metadata['content_type'] == 'text']
print(len(text_elements))

6


In [68]:
text_elements

[Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'content_type': 'text'}, page_content='Performance & Financial Statements For Three Growth-Stage Ventures  \n**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  \nFictitious Startups — Combined Annual Report FY2025'),
 Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**Executive Portfolio Overview**', 'content_type': 'text'}, page_content='This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-l

## Multi-vector retriever

**Use multi-vector retreiver to produce summaries of tables, and optionally text**.

With the summary, we'll also store the raw table elements.

The summaries are used to improve the quality of retrieval.

The raw tables are passed to the llm, providing full context for the llm to generate the answer.

**Use multi-vector retreiver to store raw tables, text along with table summaries better suited for retrieval**.


In [69]:
from langchain_core.output_parsers import StrOutputParser

In [70]:
# Prompt
prompt_text = """You are an assistant tasked with summarizing tables and text. \
               Give a concise summary of the table or text. Table or text chunk {element}"""

In [71]:
prompt = ChatPromptTemplate.from_template(prompt_text)

In [72]:
# Summary chain

summarize_chain = {"element": lambda x:x} | prompt | model | StrOutputParser()

In [76]:
table_elements

[Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**1. NovaQuant AI**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**', 'content_type': 'table'}, page_content='|**Metric (All figures in USD)**|**FY2024**|**FY2025**|**YoY Change (%)**|\n|---|---|---|---|\n|Annual Recurring Revenue (ARR)|$4,200,000|$8,450,000|+101.2%|\n|Total Revenue|$5,100,000|$9,200,000|+80.4%|\n|Cost of Goods Sold (COGS)|$1,122,000|$1,656,000|+47.6%|\n|**Gross Profit**|**$3,978,000**|**$7,544,000**|**+89.6%**|\n|Gross Margin (%)|78.0%|82.0%|+4.0%|\n|Research & Development|$2,500,000|$3,800,000|+52.0%|\n|Sales & Marketing|$1,800,000|$2,400,000|+33.3%|\n|General & Administrative|$850,000|$1,100,000|+29.4%|\n|**Net Income / (Loss)**|**($1,172,000)**|**$244,000**|**Turnaround**|'),
 Document(metadata={'Header 1': '**FY2025 ANNUAL FINANCIAL REPORT**', 'Header 2': '**2. VerdantGro Systems**', 'Header 3': '**Income Statement & Key Metrics (FY2025 vs FY2024)**', 'content_t

In [79]:
# Apply to tables
# Generate Table Summaries

tables = [ [i.metadata['Header 2'], i.page_content] for i in table_elements]

table_summaries = summarize_chain.batch(tables, {"max_concurrency":5})

In [77]:
[i.page_content for i in table_elements]

['|**Metric (All figures in USD)**|**FY2024**|**FY2025**|**YoY Change (%)**|\n|---|---|---|---|\n|Annual Recurring Revenue (ARR)|$4,200,000|$8,450,000|+101.2%|\n|Total Revenue|$5,100,000|$9,200,000|+80.4%|\n|Cost of Goods Sold (COGS)|$1,122,000|$1,656,000|+47.6%|\n|**Gross Profit**|**$3,978,000**|**$7,544,000**|**+89.6%**|\n|Gross Margin (%)|78.0%|82.0%|+4.0%|\n|Research & Development|$2,500,000|$3,800,000|+52.0%|\n|Sales & Marketing|$1,800,000|$2,400,000|+33.3%|\n|General & Administrative|$850,000|$1,100,000|+29.4%|\n|**Net Income / (Loss)**|**($1,172,000)**|**$244,000**|**Turnaround**|',
 '|**Metric (All figures in USD)**|**FY2024**|**FY2025**|**YoY Change (%)**|\n|---|---|---|---|\n|Hardware Units Shipped|1,200|3,400|+183.3%|\n|Total Revenue|$2,800,000|$6,900,000|+146.4%|\n|Cost of Goods Sold (COGS)|$1,960,000|$4,623,000|+135.9%|\n|**Gross Profit**|**$840,000**|**$2,277,000**|**+171.1%**|\n|Gross Margin (%)|30.0%|33.0%|+3.0%|\n|CapEx & Industrial Tooling|$450,000|$1,850,000|+311.1%|

In [78]:
[i.metadata['Header 2'] for i in table_elements]

['**1. NovaQuant AI**', '**2. VerdantGro Systems**', '**3. OmniHealth Go**']

Runs a Langchain batch process to summarize multiple table elements in parallel while limiting the maximum number of concurrent requests to 5.

**Batch Processing**: It takes a list of table data (tables) and passes them all at once to a summarization chain (summarize_chain).

**Concurrency Limit**: The {"max_concurrency": 5} configuration ensures that only 5 tables are processed at the same time. This prevents hitting rate limits on LLM APIs like OpenAI.

**RAG Optimization**: In multimodal or semi-structured Retrieval-Augmented Generation (RAG) pipelines, raw tables are hard to embed and search. Summarizing them creates clean text descriptions that are easier to store in a vector database for semantic search while linking back to the original table.

- Allows you to generate concise summaries of your tables.
- store those summaries in a vector database for efficient semantic search.
- retrieve the original raw tables whenever a query matches.


In [80]:
print(table_summaries[0])

Here is a concise summary of the table:

NovaQuant AI's financial performance shows significant growth in FY2025 compared to FY2024. Key highlights include:

* Annual Recurring Revenue (ARR) increased by 101.2% to $8,450,000
* Total Revenue grew by 80.4% to $9,200,000
* Gross Profit rose by 89.6% to $7,544,000, with a corresponding increase in Gross Margin from 78.0% to 82.0%
* Net Income turned around from a loss of $1,172,000 to a profit of $244,000

Overall, the company's financial performance shows strong growth and a significant turnaround in net income.


In [81]:
print(table_summaries[1])

Here is a concise summary of the table:

VerdantGro Systems reported significant growth in FY2025 compared to FY2024. Key highlights include:

* A 183.3% increase in hardware units shipped, from 1,200 to 3,400
* A 146.4% increase in total revenue, from $2.8 million to $6.9 million
* A 171.1% increase in gross profit, from $840,000 to $2.28 million
* An expansion of loss, with a net income/loss of ($560,000) in FY2024 and ($1.17 million) in FY2025

The company also saw significant increases in capital expenditures and industrial tooling, as well as operating expenses.


In [82]:
# Inspecting Text Elements

[i.page_content for i in text_elements]

['Performance & Financial Statements For Three Growth-Stage Ventures  \n**Prepared For:** Board of Directors & Stakeholders **Date of Issue:** September 2026 **Reporting Period:** January 1, 2025 – December 31, 2025 **Status:** Audited / Fictitious Reference Data  \nFictitious Startups — Combined Annual Report FY2025',
 'This aggregate annual financial report delivers an in-depth analysis of three breakout, fictitious startup ventures operating across diverse macroeconomic sectors: **NovaQuant AI** (DeepTech B2S SaaS), **VerdantGro** (AgriTech & Climate Systems), and **OmniHealth Go** (Digital Health & Direct-to-Patient Ecosystems). Despite variable market conditions throughout FY2025, the portfolio demonstrated resilient top-line growth, margin expansion, and a systematic path toward capital efficiency.',
 '**Sector:** B2B Enterprise AI & Machine Learning Infrastructure  \nNovaQuant AI specializes in high-throughput predictive financial modeling and enterprise algorithmic orchestratio

In [86]:
# Apply to text

texts = [i.page_content for i in text_elements]

text_summaries = summarize_chain.batch(texts, {"max_concurrency":5})

In [87]:
for i in range(len(text_summaries)):
    print(text_summaries[i])

Here is a concise summary of the table or text:

This report presents combined performance and financial statements for three growth-stage ventures (fictitious startups) from January 1, 2025 to December 31, 2025. The report is audited and intended for the Board of Directors and stakeholders.
Here is a concise summary of the text:

This annual financial report analyzes three startup ventures across different sectors: NovaQuant AI (DeepTech), VerdantGro (AgriTech & Climate Systems), and OmniHealth Go (Digital Health). Despite market fluctuations in FY2025, the portfolio showed strong revenue growth, improved margins, and a path to increased capital efficiency.
Here is a concise summary:

NovaQuant AI, a B2B Enterprise AI & Machine Learning Infrastructure company, achieved significant success in FY2025 by converting 45 new Fortune 500 contracts to its predictive financial modeling and algorithmic orchestration services. This shift allowed the company to focus on scalable, high-margin annu

### Creating Embeddings for the Chunks - create and persist ChromaDB Vector Store

### Add to vectorstore 

Use multi vector retriever with summaries.

- **InMemoryStore** - stores the raw text, tables.
- **vectorstore** - stores the embedded summaries.

In [88]:
import uuid
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiVectorRetriever

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

#from langchain_community.vectorstores import Chroma

In [89]:
# Embedding model of your choice
embedding_model = OllamaEmbeddings(model = "llama3.2")

In [90]:
# Vector store to use to index the child chunks.
# Vector store for the searchable child/summary vectors.


vectorstore = Chroma(collection_name="summaries",
                     embedding_function = embedding_model)

In [91]:
# Storage layer for the full parent documents

store = InMemoryStore()
id_key = "doc_id"

In [92]:
# Retriever (empty to start)
# Initialize the MultiVectorRetriever

retriever_multi = MultiVectorRetriever(vectorstore=vectorstore,
                                       docstore = store,
                                       id_key = id_key,
                                       )
# id_key - the one that we just created above - in memory store - parent documents.

In [96]:
# Add texts

doc_ids = [str(uuid.uuid4()) for _ in texts]

summary_texts = [Document(page_content=s,metadata={id_key:doc_ids[i]}) for i,s in enumerate(text_summaries)]
# llm generated summary becomes page content

retriever_multi.vectorstore.add_documents(summary_texts)
retriever_multi.docstore.mset(list(zip(doc_ids, texts)))

In [97]:
# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]

# Package the summaries as Documents with metadata linking them back to the ID
summary_tables = [Document(page_content=s,metadata={id_key:table_ids[i]}) for i,s in enumerate(table_summaries)]

# Add data to the retriever
retriever_multi.vectorstore.add_documents(summary_tables) # Adds summaries to Chroma
retriever_multi.docstore.mset(list(zip(table_ids, tables))) # Adds raw tables to InMemoryStore


How This Works Behind the Scenes

**The Core Search Engine (vectorstore)**

- The vector database only indexes the short summaries generated by your batch process. 
- This avoids polluting the vector space with raw markdown code or disjointed table figures that confuse distance metrics.

**The Document Parent Vault (docstore)**

- The original, raw markdown tables are stored in a key-value store (InMemoryStore) mapping a unique doc_id to the table text.

**The Intermediary (MultiVectorRetriever)**

- When you call retriever.invoke(query), it scans the vector database for the closest summary match. Once it finds the winning summary, it grabs the doc_id from its metadata, looks up that ID inside the docstore, and hands you back the raw table to feed directly into your final generation LLM prompt.

**Architecture** - Swap out InMemoryStore for a persistent alternative like Redis or a SQL database to keep data across restarts.

Build a hybrid Multi-Vector Retriever for both files and tables.

Expanded architecture that handles both text chunks and tables together.

In this hybrid setup, text chunks are stored and searched directly, while tables are summarized first—with the search pointing to the summaries but returning the original tables. Both types of documents co-exist seamlessly inside the same MultiVectorRetriever.

### RETRIEVAL 

In [98]:
# Querying will search against the summaries but return the original markdown table

# Query the retriever (searches summaries, returns full parent docs)

query = "Show me the revenue and gross profit for novaquant"
retrieved_docs = retriever_multi.invoke(query)

In [99]:
print(retrieved_docs)

['**Sector:** B2B Enterprise AI & Machine Learning Infrastructure  \nNovaQuant AI specializes in high-throughput predictive financial modeling and enterprise algorithmic orchestration. In FY2025, NovaQuant successfully converted 45 new Fortune 500 contracts, shifting operational focus away from capital-intensive professional services toward scalable, high-margin annual recurring subscription tiers.', '**Sector:** AgriTech / Climate Tech Hardware-enabled SaaS  \nVerdantGro manufactures Automated Hydroponic Canopy Nodes deployed alongside AI-optimized resource monitoring portals. FY2025 marked a significant scale-up phase with heavy investments into domestic production facilities, triggering a short-term drop in net profitability despite booking explosive  \nConfidential — For Internal & Investor Review Only  \nPage 2 of 4  \nFictitious Startups — Combined Annual Report FY2025  \n#### volume gains in physical unit distribution.', 'This aggregate annual financial report delivers an in-dep

### ANSWER GENERATION

In [102]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

In [103]:
# The prompt instructs the LLM to handle both narrative paragraphs and raw tables.

template = """You are an expert financial and data analyst assistant. 
Answer the question based strictly on the provided context, which can include text and tables. 
If the context contains a table, interpret its rows and columns carefully to answer accurately. 
If you do not know the answer based on the context, state that you do not know.

Context:
{context}

Question: {question}
"""

In [104]:
prompt = ChatPromptTemplate.from_template(template)

In [105]:
# RAG Pipeline

chain = (
    {"context":retriever_multi, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
# Example 1: Testing an analytical query targeting a raw table structure
query_1 = "What are the startups?"
print(f"User Question: {query_1}\n")
response_1 = chain.invoke(query_1)
print(f"LLM Response:\n{response_1}")

print("\n" + "="*50 + "\n")

User Question: What are the startups?

LLM Response:
Based on the provided context, the three startups mentioned are:

1. **NovaQuant AI**: Specializes in high-throughput predictive financial modeling and enterprise algorithmic orchestration.
2. **VerdantGro**: Manufactures Automated Hydroponic Canopy Nodes deployed alongside AI-optimized resource monitoring portals.
3. **OmniHealth Go**: Operates in the Digital Health & Direct-to-Patient Ecosystems sector.

These startups are part of a combined annual report for FY2025, which provides an overview of their financial performance and growth.




In [107]:
query_2 = "Show me the gross profit for novaquant"
print(f"User Question: {query_2}\n")
response_2 = chain.invoke(query_2)
print(f"LLM Response:\n{response_2}")

print("\n" + "="*50 + "\n")

User Question: Show me the gross profit for novaquant

LLM Response:
I don't have information about NovaQuant's financials in the provided context. The table only shows data for VerdantGro Systems, and there is no mention of NovaQuant's financial metrics. Therefore, I do not know the answer to this question.




In [108]:
query_3 = "What is the gross profit for novaquant"
print(f"User Question: {query_3}\n")
response_3 = chain.invoke(query_3)
print(f"LLM Response:\n{response_3}")

print("\n" + "="*50 + "\n")

User Question: What is the gross profit for novaquant

LLM Response:
I don't know the answer to this question based on the provided context because there is no information about NovaQuant's financial performance in the table or text. The table only provides data for VerdantGro, and there is no mention of NovaQuant's financial metrics.




In [109]:
query_4 = "What is the revenue for novaquant?"
print(f"User Question: {query_4}\n")
response_4 = chain.invoke(query_4)
print(f"LLM Response:\n{response_4}")

print("\n" + "="*50 + "\n")

User Question: What is the revenue for novaquant?

LLM Response:
According to the table provided, NovaQuant AI's Total Revenue in FY2024 was $5,100,000 and in FY2025 it was $9,200,000. Therefore, the revenue for NovaQuant AI is $9,200,000.




In [110]:
query_5 = "What is the gross profit for novaquant?"
print(f"User Question: {query_5}\n")
response_5 = chain.invoke(query_5)
print(f"LLM Response:\n{response_5}")

print("\n" + "="*50 + "\n")

User Question: What is the gross profit for novaquant?

LLM Response:
According to the provided table, the Gross Profit for NovaQuant AI in FY2024 was $3,978,000 and in FY2025 it was $7,544,000.

So, the answer is: **$7,544,000**.




In [111]:
query_6 = "What is the gross profit for all the startups?"
print(f"User Question: {query_6}\n")
response_6 = chain.invoke(query_6)
print(f"LLM Response:\n{response_6}")

print("\n" + "="*50 + "\n")

User Question: What is the gross profit for all the startups?

LLM Response:
Based on the provided context, specifically the table for **OmniHealth Go**, we can see that:

|**Metric (All figures in USD)**|**FY2024**|**FY2025**|**YoY Change (%)**|
|---|---|---|---|
|**Gross Profit**|**$2,340,000**|**$5,070,000**|**+116.7%**

The gross profit for **OmniHealth Go** in FY2024 is $2,340,000 and in FY2025 it is $5,070,000.

Since there are no other startups mentioned in the table, we can conclude that the gross profit for all the startups (i.e., **NovaQuant AI**, **VerdantGro**, and **OmniHealth Go**) is:

$2,340,000 + ? = ?

Unfortunately, there is no information provided about the gross profits of **NovaQuant AI** and **VerdantGro**, so I do not know the answer to this question.




In [112]:
query_7 = "Which startup has the highest revenue?"
print(f"User Question: {query_7}\n")
response_7 = chain.invoke(query_7)
print(f"LLM Response:\n{response_7}")

print("\n" + "="*50 + "\n")

User Question: Which startup has the highest revenue?

LLM Response:
Based on the provided context and table, I can answer that **NovaQuant AI** has the highest revenue in FY2025, with a total revenue of $[insert value], which is not specified in the given data. However, we do have information about the other two startups: OmniHealth Go had a total revenue of $7,800,000, and VerdantGro's revenue is not mentioned at all.

So, to answer your question directly: **OmniHealth Go** has the highest revenue among the three startups mentioned in FY2025.




In [113]:
query_8 = "What are the financial metrics for novaquant, can you show me in a tabular format?"
print(f"User Question: {query_8}\n")
response_8 = chain.invoke(query_8)
print(f"LLM Response:\n{response_8}")

print("\n" + "="*50 + "\n")

User Question: What are the financial metrics for novaquant, can you show me in a tabular format?

LLM Response:
Based on the provided context, I can extract the financial metrics for NovaQuant AI and present them in a tabular format:

**NovaQuant AI Financial Metrics**

| **Metric (All figures in USD)** | **FY2024** | **FY2025** | **YoY Change (%)** |
| --- | --- | --- | --- |
| Annual Recurring Revenue (ARR) | $4,200,000 | $8,450,000 | +101.2% |
| Total Revenue | $5,100,000 | $9,200,000 | +80.4% |
| Cost of Goods Sold (COGS) | $1,122,000 | $1,656,000 | +47.6% |
| Gross Profit | $3,978,000 | $7,544,000 | +89.6% |
| Gross Margin (%) | 78.0% | 82.0% | +4.0% |
| Research & Development | $2,500,000 | $3,800,000 | +52.0% |
| Sales & Marketing | $1,800,000 | $2,400,000 | +33.3% |
| General & Administrative | $850,000 | $1,100,000 | +29.4% |
| Net Income / (Loss) | ($1,172,000) | $244,000 | Turnaround |

Please note that these metrics are specific to NovaQuant AI and do not represent the com

In [114]:
query_9 = "What is the percentage change in r&d for novaquant?"
print(f"User Question: {query_9}\n")
response_9 = chain.invoke(query_9)
print(f"LLM Response:\n{response_9}")

print("\n" + "="*50 + "\n")

User Question: What is the percentage change in r&d for novaquant?

LLM Response:
Based on the provided context, specifically the table for NovaQuant AI, I can answer that:

The YoY (Year-over-Year) Change (%) for Research & Development (R&D) is +52.0%.

So, the percentage change in R&D for NovaQuant is 52%.




In [115]:
query_10 = "What does the investment in R&D look like for all companies?"
print(f"User Question: {query_10}\n")
response_10 = chain.invoke(query_10)
print(f"LLM Response:\n{response_10}")

print("\n" + "="*50 + "\n")

User Question: What does the investment in R&D look like for all companies?

LLM Response:
I don't know. The provided context only includes information about OmniHealth Go, NovaQuant AI, and VerdantGro's financial performance, but it doesn't mention anything about Research and Development (R&D) investments or expenditures. Therefore, I cannot provide an answer regarding the investment in R&D for all companies.




In [116]:
query_11 = "What are the financial metrics for all the companies, can you show me in a tabular format?"
print(f"User Question: {query_11}\n")
response_11 = chain.invoke(query_11)
print(f"LLM Response:\n{response_11}")

print("\n" + "="*50 + "\n")

User Question: What are the financial metrics for all the companies, can you show me in a tabular format?

LLM Response:
Based on the provided context, I can extract the financial metrics for both NovaQuant AI and OmniHealth Go. Here is the combined table:

| **Metric (All figures in USD)** | **FY2024** | **FY2025** | **YoY Change (%)** |
| --- | --- | --- | --- |
| Active Monthly Subscribers | - | 112,000 | +148.9% |
| Total Revenue | $3,900,000 | $7,800,000 | +100.0% |
| Cost of Goods Sold (COGS) | $1,560,000 | $2,730,000 | +75.0% |
| Gross Profit | $2,340,000 | $5,070,000 | +116.7% |
| Gross Margin (%) | 60.0% | 65.0% | +5.0% |
| Customer Acquisition (CAC) | $1,100,000 | $1,950,000 | +77.3% |
| Administrative Overhead | $780,000 | $1,200,000 | +53.8% |
| Net Income / (Loss) | $460,000 | $1,920,000 | +317.4% |

**NovaQuant AI**

| **Metric (All figures in USD)** | **FY2024** | **FY2025** | **YoY Change (%)** |
| --- | --- | --- | --- |
| Annual Recurring Revenue (ARR) | $4,200,000 | 

In [117]:
query_12 = "Can you provide a brief description of the startups?"
print(f"User Question: {query_12}\n")
response_12 = chain.invoke(query_12)
print(f"LLM Response:\n{response_12}")

print("\n" + "="*50 + "\n")

User Question: Can you provide a brief description of the startups?

LLM Response:
Based on the provided context, here is a brief description of the three startups:

* **NovaQuant AI**: A B2B enterprise AI & machine learning infrastructure company that specializes in high-throughput predictive financial modeling and enterprise algorithmic orchestration. They successfully converted 45 new Fortune 500 contracts in FY2025.
* **VerdantGro**: An AgriTech / Climate Tech company that manufactures Automated Hydroponic Canopy Nodes deployed alongside AI-optimized resource monitoring portals. In FY2025, they experienced a scale-up phase with heavy investments into domestic production facilities, leading to a short-term drop in net profitability.

Note: There is no information provided about **OmniHealth Go**, except for its financial metrics in the table.


